-------------------------------------------------------------------------
*   PONTIFÍCIA UNIVERSIDADE CATÓLICA DE MINAS GERAIS
*   PROFESSOR: VICTOR SALES SILVA
*   ALUNO: DGEISON SERRÃO PEIXOTO
*   MATRÍCULA: **1366415**
*   ATIVIDADE: LEITURA DE ARQUIVO EM FORMATO XML UTILIZANDO SPARK
-------------------------------------------------------------------------

# INSTALAÇÃO DAS BIBLIOTECAS

In [1]:
%pip install pyspark
!pip install azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.9/412.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 14.3 MB/s eta 0:00:00


# IMPORTAÇÃO DAS BIBLIOTECAS

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import TimestampType, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col
from pyspark.sql.functions import to_date
import xml.etree.ElementTree as ET
from azure.storage.blob import BlobServiceClient
from azure.storage.blob import BlobClient

# CRIAÇÃO DA APLICAÇÃO SPARK

In [3]:
spark = SparkSession.builder.getOrCreate()

# VARIÁVEIS DE APOIO

In [4]:
storageaccount = 'stgaccount687878'
container = 'datalake-687878'
connection_string = 'DefaultEndpointsProtocol=https;AccountName=stgaccount687878;AccountKey=SUA_ACCOUNT_KEY_AQUI;EndpointSuffix=core.windows.net'
blob_file = 'bronze/DADOS_ALUNOS/DADOS_ALUNOS.xml'

In [5]:
def listar_arquivos_no_container(conn_string, container_name, prefixo=""):
  try:
      # 1. Conecta ao serviço de Blob
      blob_service_client = BlobServiceClient.from_connection_string(conn_string)

      # 2. Obtém o cliente para o contêiner
      container_client = blob_service_client.get_container_client(container_name)

      print(f"Buscando arquivos em '{container_name}' com o prefixo '{prefixo}'...")

      # 3. Lista os blobs (arquivos) que começam com o prefixo
      blob_list = container_client.list_blobs(name_starts_with=prefixo)

      lista_de_arquivos = []
      for blob in blob_list:
          print(f"  - {blob.name}")
          lista_de_arquivos.append(blob.name)

      if not lista_de_arquivos:
          print("\nNenhum arquivo encontrado neste caminho.")

      return lista_de_arquivos

  except Exception as e:
      print(f"Ocorreu um erro ao tentar listar os arquivos: {e}")
      return []


camada_para_verificar = 'bronze/'

print(f"--- Verificando o conteúdo da camada '{camada_para_verificar}' ---")
arquivos_encontrados = listar_arquivos_no_container(
    connection_string,
    container,
    prefixo=camada_para_verificar
)

print("\n--- Fim da verificação ---")

--- Verificando o conteúdo da camada 'bronze/' ---
Buscando arquivos em 'datalake-687878' com o prefixo 'bronze/'...
  - bronze/DADOS_ALUNOS/DADOS_ALUNOS.xml
  - bronze/DADOS_BANCARIOS/DADOS_BANCARIOS.xml
  - bronze/DADOS_ESTUDANTES/DADOS_ESTUDANTES.json
  - bronze/DADOS_EXAMES/DADOS_EXAMES.csv
  - bronze/DADOS_VOOS/DADOS_VOOS.parquet

--- Fim da verificação ---


# FUNÇÃO PARA LER ARQUIVO XML

In [6]:
blob_file_na_nuvem = 'bronze/DADOS_BANCARIOS/DADOS_BANCARIOS.xml'
arquivo_local = 'DADOS_BANCARIOS.xml'

print(f"Baixando o arquivo '{blob_file_na_nuvem}' da nuvem...")
try:
    blob_client = BlobClient.from_connection_string(
        conn_str=connection_string,
        container_name=container,
        blob_name=blob_file_na_nuvem
    )
    with open(arquivo_local, "wb") as my_blob:
        blob_data = blob_client.download_blob()
        blob_data.readinto(my_blob)

    print(f"Arquivo salvo localmente como '{arquivo_local}' com sucesso!")

except Exception as e:
    print(f"Ocorreu um erro no download: {e}")
    arquivo_local = None




Baixando o arquivo 'bronze/DADOS_BANCARIOS/DADOS_BANCARIOS.xml' da nuvem...
Arquivo salvo localmente como 'DADOS_BANCARIOS.xml' com sucesso!


In [7]:
def ler_xml(arquivo):
    tree = ET.parse(arquivo_local)
    root = tree.getroot()
    return root

# CRIAÇÃO DE LISTA COM O CONTEÚDO DO XML

In [8]:
root = ler_xml(arquivo_local)
dados_bancarios = []

for record in root:
  for record_user_info in record.findall('user_info'):
    name = record_user_info.find('name').text
    address = record_user_info.find('address').text
    gender = record_user_info.find('gender').text
    account_opening_date = record_user_info.find('account_opening_date').text
    customer_type = record_user_info.find('customer_type').text
    date_of_birth = record_user_info.find('date_of_birth').text
  for record_account_info in record.findall('account_info'):
    account_number = record_account_info.find('account_number').text
    account_type = record_account_info.find('account_type').text
    balance = record_account_info.find('balance').text
    currency = record_account_info.find('currency').text
    branch = record_account_info.find('branch').text
  for record_transaction_info in record.findall('transactions/'):
    date = record_transaction_info.find('date').text
    description = record_transaction_info.find('description').text
    amount = record_transaction_info.find('amount').text
    dados_bancarios.append([name, address, gender, account_opening_date, customer_type, date_of_birth, account_number, account_type, balance, currency, branch, date, description, amount])

# LEITURA DA LISTA USANDO SPARK

In [9]:
df = spark.createDataFrame(dados_bancarios, ['name', 'address', 'gender', 'account_opening_date', 'customer_type', 'date_of_birth', 'account_number', 'account_type', 'balance', 'currency', 'branch', 'date', 'description', 'amount'])

# EXIBINDO UMA AMOSTRA DOS DADOS

In [10]:
df.show(truncate=False)

+-----------------+--------------------------------------------------+------+--------------------+-------------+-------------+----------------------+------------+--------+--------+------+----------+----------------------------------------------------+------+
|name             |address                                           |gender|account_opening_date|customer_type|date_of_birth|account_number        |account_type|balance |currency|branch|date      |description                                         |amount|
+-----------------+--------------------------------------------------+------+--------------------+-------------+-------------+----------------------+------------+--------+--------+------+----------+----------------------------------------------------+------+
|Justin Sullivan  |95954 Manuel Viaduct\nGarciaberg, OK 85968        |Male  |2018-11-07          |Basic        |1946-08-03   |GB78UOLI44528467272790|Savings     |49774.73|LTL     |63    |2022-07-16|Streamlined tangible appl

# EXIBINDO OS METADADOS (SCHEMA) DO ARQUIVO

In [11]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- account_opening_date: string (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- account_number: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- balance: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- date: string (nullable = true)
 |-- description: string (nullable = true)
 |-- amount: string (nullable = true)



# AJUSTANDO O SCHEMA DOS DADOS, SE NECESSÁRIO

In [12]:
df = df.withColumn('amount', df['amount'].cast('double'))
df = df.withColumn('balance ', df['balance'].cast('double'))
df = df.withColumn("account_opening_date", df['account_opening_date'].cast('date'))
df = df.withColumn("date_of_birth", df['date_of_birth'].cast('date'))
df = df.withColumn("date", df['date'].cast('date'))
df = df.withColumn('branch', df['branch'].cast('long'))

In [14]:
df.show(truncate=False)

+-----------------+--------------------------------------------------+------+--------------------+-------------+-------------+----------------------+------------+--------+--------+------+----------+----------------------------------------------------+------+--------+
|name             |address                                           |gender|account_opening_date|customer_type|date_of_birth|account_number        |account_type|balance |currency|branch|date      |description                                         |amount|balance |
+-----------------+--------------------------------------------------+------+--------------------+-------------+-------------+----------------------+------------+--------+--------+------+----------+----------------------------------------------------+------+--------+
|Justin Sullivan  |95954 Manuel Viaduct\nGarciaberg, OK 85968        |Male  |2018-11-07          |Basic        |1946-08-03   |GB78UOLI44528467272790|Savings     |49774.73|LTL     |63    |2022-07-1